# On-Call Incident Response Memory (Strands + Episodic + Metadata)

## Introduction

This tutorial builds an SRE / on-call assistant that turns every incident-response conversation into a **queryable post-mortem record** using `EpisodicMemoryStrategy` with a custom metadata schema.

The schema is grounded in real incident-response taxonomy — **SEV1–SEV5 severity**, **services touched** (STRINGLIST), **root-cause category**, and **mitigation applied**. The agent answers questions like *"what did we do last time we had a race condition in auth-service that we rolled back?"* — a question that is essentially impossible to answer with semantic search alone.

### Tutorial Details

| Information         | Details                                                                    |
|:--------------------|:---------------------------------------------------------------------------|
| Tutorial type       | Long-term memory with metadata filtering (episodic)                        |
| Agent type          | On-call / incident-response coding assistant                               |
| Agentic Framework   | Strands Agents                                                             |
| LLM model           | Anthropic Claude Haiku 4.5                                                 |
| Tutorial components | Episodic strategy with `metadataSchema`, NUMBER + STRINGLIST indexed keys |
| Example complexity  | Intermediate                                                               |

### You'll learn to
- Attach a `metadataSchema` to an **episodic** strategy (not just semantic)
- Rely on **implicit FM extraction** for everything — no engineer tagging at event time
- Combine `LESS_THAN_OR_EQUALS` (NUMBER) with `CONTAINS` (STRINGLIST) and two `EQUALS_TO` filters in one retrieval
- Use `listMemoryRecords` without semantic search for deterministic audit enumeration

## Prerequisites
- Python 3.10+
- AWS credentials with `bedrock-agentcore` and `bedrock-agentcore-control` permissions
- A `memory_execution_role_arn`
- Amazon Bedrock access to Anthropic Claude Haiku 4.5


## Step 1: Install Dependencies

In [ ]:
!pip install -qr requirements.txt

## Step 2: Imports and Configuration

In [ ]:
import logging
import time
import json
import uuid
from datetime import datetime, timedelta, timezone
from typing import List, Dict, Optional

import boto3
from botocore.exceptions import ClientError

from strands import Agent, tool

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("oncall-metadata")
logger.info("Imports loaded")


In [ ]:
# Replace with your values
REGION = "us-west-2"
MEMORY_EXECUTION_ROLE_ARN = "arn:aws:iam::<ACCOUNT_ID>:role/<AgentCoreMemoryExecutionRole>"

ENGINEER_ID = "oncall-eng-001"
SESSION_ID = f"oncall_{datetime.now().strftime('%Y%m%d%H%M%S')}"

logger.info(f"Region:   {REGION}")
logger.info(f"Engineer: {ENGINEER_ID}")
logger.info(f"Session:  {SESSION_ID}")


## Step 3: Create Memory with Episodic Strategy + Metadata Schema

Episodic strategies preserve *complete interaction sequences* (the full incident conversation) rather than just isolated facts. We layer a `metadataSchema` on top so that each extracted episode carries structured incident attributes:

- **`incident_severity`** (NUMBER 1–5): SRE's SEV1–SEV5 scale, where **lower is worse**.
- **`services_touched`** (STRINGLIST): FM pulls microservice names mentioned in the incident.
- **`root_cause_category`** (STRING): constrained vocabulary of real failure modes.
- **`mitigation_applied`** (STRING): constrained vocabulary of what actually worked.
- **`mttr_estimate_hours`** (NUMBER, **non-indexed**): schema-only enrichment for reporting.

Notice every key's extraction is driven purely by the conversation — we supply **no event metadata**. This demonstrates the "implicit extraction" pattern from the blog: schema keys without matching event metadata get their values entirely from the FM's reading of the conversation.


In [ ]:
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

memory_name = "OnCallIncidentMetadataMemory"

indexed_keys = [
    {"key": "incident_severity",   "type": "NUMBER"},
    {"key": "services_touched",    "type": "STRING_LIST"},
    {"key": "root_cause_category", "type": "STRING"},
    {"key": "mitigation_applied",  "type": "STRING"},
    # Note: mttr_estimate_hours is NOT indexed
]

metadata_schema = [
    {
        "key": "incident_severity",
        "type": "NUMBER",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "SEV1-SEV5 incident severity where 1 = catastrophic (customer-facing total outage), "
                    "2 = major (significant degradation), 3 = moderate (single-service impact), "
                    "4 = minor (internal only), 5 = trivial (cosmetic). Lower values are worse."
                ),
                "llmExtractionInstruction": (
                    "Infer from blast radius and urgency signals in the conversation. "
                    "Default to the highest (worst) severity referenced. Output an integer 1-5."
                ),
                "validation": {
                    "numericValidation": {"minimum": 1, "maximum": 5}
                }
            }
        }
    },
    {
        "key": "services_touched",
        "type": "STRING_LIST",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Names of microservices, databases, or named infrastructure components "
                    "explicitly discussed as involved in or affected by the incident."
                ),
                "llmExtractionInstruction": (
                    "Extract all referenced service/component names as a list. "
                    "Normalize to kebab-case (e.g. 'auth-service', 'payment-db', 'frontend-cdn'). "
                    "Do not include generic terms ('database', 'the backend'). Dedupe."
                )
            }
        }
    },
    {
        "key": "root_cause_category",
        "type": "STRING",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "The established root cause category. Use memory_leak for OOM/heap issues, "
                    "race_condition for concurrency bugs, config_drift for env/config mismatches, "
                    "dependency_mismatch for version/library conflicts, data_corruption for bad data, "
                    "capacity for resource exhaustion, external_dependency for 3rd-party outages."
                ),
                "llmExtractionInstruction": "LATEST_VALUE",
                "validation": {
                    "stringValidation": {
                        "allowedValues": [
                            "memory_leak", "race_condition", "config_drift",
                            "dependency_mismatch", "data_corruption", "capacity",
                            "external_dependency"
                        ]
                    }
                }
            }
        }
    },
    {
        "key": "mitigation_applied",
        "type": "STRING",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "The decisive action that actually mitigated the incident. "
                    "revert = revert code, hotfix = patch forward, rollback = redeploy prior version, "
                    "config_change = env/config edit, scale_up = add capacity, no_action = self-healed."
                ),
                "llmExtractionInstruction": "LATEST_VALUE",
                "validation": {
                    "stringValidation": {
                        "allowedValues": [
                            "revert", "hotfix", "rollback", "config_change", "scale_up", "no_action"
                        ]
                    }
                }
            }
        }
    },
    {
        "key": "mttr_estimate_hours",
        "type": "NUMBER",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Estimated mean-time-to-mitigate in hours. From incident detection to mitigation "
                    "applied. Fractional hours allowed."
                ),
                "llmExtractionInstruction": "Infer from timestamps and explicit duration mentions."
            }
        }
    },
]

episodic_strategy = {
    "episodicMemoryStrategy": {
        "name": "IncidentEpisodeExtractor",
        "description": "Incident response episodes with SRE metadata",
        "namespaceTemplates": ["/oncall/{actorId}/"],
        "memoryRecordSchema": {"metadataSchema": metadata_schema}
    }
}

try:
    resp = control_client.create_memory(
        name=memory_name,
        eventExpiryDuration=90,
        memoryExecutionRoleArn=MEMORY_EXECUTION_ROLE_ARN,
        indexedKeys=indexed_keys,
        memoryStrategies=[episodic_strategy],
        clientToken=str(uuid.uuid4()),
    )
    memory_id = resp["memory"]["id"]
    logger.info(f"✅ Created memory {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        existing = control_client.list_memories()["memories"]
        match = next((m for m in existing if m["name"] == memory_name), None)
        memory_id = match["id"]
        logger.info(f"ℹ️  Reusing existing memory {memory_id}")
    else:
        raise

namespace = f"/oncall/{ENGINEER_ID}/"
logger.info(f"Namespace: {namespace}")


## Step 4: Wait for Memory to be Active


In [ ]:
def wait_active(mem_id, timeout_s=240):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        m = control_client.get_memory(memoryId=mem_id)["memory"]
        logger.info(f"status: {m['status']}")
        if m["status"] == "ACTIVE":
            return
        if m["status"] == "FAILED":
            raise RuntimeError(f"memory failed: {m.get('failureReason')}")
        time.sleep(10)
    raise TimeoutError("memory never reached ACTIVE")

wait_active(memory_id)


## Step 5: Mock SRE Tools

Realistic on-call tooling so the agent feels grounded in the domain. These tools are mocks — in production they would call Datadog, PagerDuty, your deploy system, etc.


In [ ]:
@tool
def check_service_health(service: str) -> str:
    """Return the current health status of a named service."""
    statuses = {
        "auth-service":   "degraded — 22% error rate, 800ms p99",
        "payment-db":     "healthy",
        "frontend-cdn":   "healthy",
        "notification-svc": "healthy",
    }
    return statuses.get(service, f"unknown service: {service}")

@tool
def list_recent_deploys(service: str, hours: int = 24) -> str:
    """List deploys to a service in the last N hours."""
    deploys = {
        "auth-service": [
            "14:02 UTC — v2.3.1 — @alice — session refactor",
            "09:30 UTC — v2.3.0 — @alice — dependency bump",
        ],
        "payment-db": ["yesterday 20:10 UTC — schema migration m0044"],
    }
    return "\n".join(deploys.get(service, [f"no deploys in the last {hours}h"]))

@tool
def page_oncall(team: str, reason: str) -> str:
    """Page an on-call team. Mocked."""
    return f"paged {team}: {reason}"

logger.info("✅ Tools defined")


## Step 6: Seed Three Historical Incidents

Each incident is a full conversation representing what actually happened during the response. We submit them **with no event metadata** — all five schema fields will be inferred from the conversation content by the episodic extractor.


In [ ]:
data_client = boto3.client("bedrock-agentcore", region_name=REGION)

def submit_incident(session_suffix: str, turns, backdate_hours: int):
    session_id = f"incident-{session_suffix}"
    base_ts = datetime.now(timezone.utc) - timedelta(hours=backdate_hours)
    payload = []
    for i, (role, text) in enumerate(turns):
        payload.append({
            "conversational": {
                "role": "USER" if role == "user" else "ASSISTANT",
                "content": {"text": text},
            }
        })
    data_client.create_event(
        memoryId=memory_id,
        actorId=ENGINEER_ID,
        sessionId=session_id,
        eventTimestamp=base_ts,
        payload=payload,
    )
    logger.info(f"submitted incident {session_suffix}")

# INCIDENT A — SEV1 race condition in auth-service, rolled back
submit_incident("a-auth-race", [
    ("user",      "Auth-service throwing 401s for ~30% of users. Started right after the v2.3.1 deploy at 14:02 UTC. This is SEV1."),
    ("assistant", "Pulling logs and looking for shared-state issues after v2.3.1."),
    ("user",      "Looks like a race on the session cache invalidation — concurrent writes are corrupting entries."),
    ("assistant", "Confirmed. Two code paths are writing to the session cache without a lock. Classic race condition."),
    ("user",      "Can't hotfix safely — let's rollback to v2.3.0."),
    ("assistant", "Rolling back auth-service to v2.3.0. Error rate normalizing. Total impact ~28 minutes."),
    ("user",      "Good. We'll write a proper fix tomorrow with a distributed lock."),
    ("assistant", "Incident closed. SEV1, auth-service, race condition, rollback applied."),
], backdate_hours=72)

# INCIDENT B — SEV2 memory leak in payment-db, hotfixed
submit_incident("b-payment-leak", [
    ("user",      "Payment-db memory grew from 40% to 95% over 4 hours. SEV2 — payments slowing but not down."),
    ("assistant", "Heap growing linearly — looks like a memory leak. Checking connection-pool stats."),
    ("user",      "Prepared statements aren't being released. The recent orm upgrade is holding references."),
    ("assistant", "Found it — a finalizer was removed in orm 3.2 that previously closed unused statements."),
    ("user",      "Hotfix it — we can't rollback, there was a schema change that doesn't unmigrate cleanly."),
    ("assistant", "Applying hotfix — explicit stmt.close() in the repo layer. Memory stabilized at 60%."),
    ("user",      "Total time ~5 hours from first pages."),
    ("assistant", "Incident closed. SEV2, payment-db, memory_leak, hotfix applied."),
], backdate_hours=48)

# INCIDENT C — SEV3 config drift in frontend-cdn, config change
submit_incident("c-cdn-config", [
    ("user",      "CSS missing on frontend after CDN config update. SEV3 — degraded UX but site works."),
    ("assistant", "Checking cdn config. Cache-control header was changed in last night's update."),
    ("user",      "Yeah, the TTL was set to 0 for *.css by mistake."),
    ("assistant", "Fixing cdn config to the prior TTL value. Change deployed."),
    ("user",      "Looks good now. ~90 min total."),
    ("assistant", "Incident closed. SEV3, frontend-cdn, config_drift, config_change."),
], backdate_hours=24)

logger.info("All 3 incidents submitted")


## Step 7: Wait for Episodic Extraction

Episode extraction takes a minute or two. Poll until we see records.


In [ ]:
def wait_records(expected_min=2, timeout_s=240):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        r = data_client.list_memory_records(memoryId=memory_id, namespace=namespace, maxResults=50)
        recs = r.get("memoryRecordSummaries", [])
        logger.info(f"records so far: {len(recs)}")
        if len(recs) >= expected_min:
            return recs
        time.sleep(15)
    raise TimeoutError(f"only {len(recs)} records after {timeout_s}s")

records = wait_records(expected_min=3)
logger.info(f"✅ {len(records)} records extracted")


## Step 8: Inspect FM-Inferred Metadata

Confirm the extractor populated every field from conversation content alone. You should see `services_touched` as a list, `incident_severity` as a number in [1,5], and the constrained-vocabulary STRING fields.


In [ ]:
for i, r in enumerate(records, 1):
    full = data_client.get_memory_record(memoryId=memory_id, memoryRecordId=r["memoryRecordId"])["memoryRecord"]
    print(f"\n=== Episode #{i} ({full['memoryRecordId']}) ===")
    text = full.get("content", {}).get("text", "")
    print(f"content preview: {text[:180]}")
    for k, v in full.get("metadata", {}).items():
        print(f"  {k}: {v}")


## Step 9: Retrieval — Operational Queries

### Query 1 — service-scoped retrieval

*"What did we do last time `auth-service` had a problem?"*


In [ ]:
def retrieve(query, filters=None, top_k=10):
    params = {"searchQuery": query, "topK": top_k}
    if filters:
        params["metadataFilters"] = filters
    resp = data_client.retrieve_memory_records(
        memoryId=memory_id, namespace=namespace, searchCriteria=params
    )
    return resp.get("memoryRecordSummaries", [])

def show(results, label):
    print(f"\n--- {label}: {len(results)} results ---")
    for r in results:
        print(f"  [{r.get('score', 0):.3f}] {r.get('content', {}).get('text', '')[:160]}")

auth_issues = retrieve(
    "what did we do when auth broke",
    filters=[
        {"left": {"metadataKey": "services_touched"}, "operator": "CONTAINS",
         "right": {"metadataValue": {"stringValue": "auth-service"}}},
    ],
)
show(auth_issues, "services_touched CONTAINS 'auth-service'")


### Query 2 — high-severity filter

*"Just give me SEV1–SEV2 incidents."*


In [ ]:
sev_filter = [
    {"left": {"metadataKey": "incident_severity"}, "operator": "LESS_THAN_OR_EQUALS",
     "right": {"metadataValue": {"numberValue": 2}}},
]
sev_results = retrieve("recent major incidents", filters=sev_filter)
show(sev_results, "incident_severity <= 2")


### Query 3 — compound filter

*"SEV≤2 race-condition incidents in auth-service where we rolled back."* Four filters combined with AND.


In [ ]:
compound = [
    {"left": {"metadataKey": "incident_severity"}, "operator": "LESS_THAN_OR_EQUALS",
     "right": {"metadataValue": {"numberValue": 2}}},
    {"left": {"metadataKey": "root_cause_category"}, "operator": "EQUALS_TO",
     "right": {"metadataValue": {"stringValue": "race_condition"}}},
    {"left": {"metadataKey": "services_touched"}, "operator": "CONTAINS",
     "right": {"metadataValue": {"stringValue": "auth-service"}}},
    {"left": {"metadataKey": "mitigation_applied"}, "operator": "EQUALS_TO",
     "right": {"metadataValue": {"stringValue": "rollback"}}},
]
comp_results = retrieve("how did we mitigate", filters=compound)
show(comp_results, "SEV<=2 AND race_condition AND auth-service AND rollback")


### Query 4 — unfiltered baseline

Same *"how did we mitigate"* query with **no filters**. Compare the relevance and count.


In [ ]:
unfiltered = retrieve("how did we mitigate")
show(unfiltered, "UNFILTERED: 'how did we mitigate'")


## Step 10: Non-Semantic Enumeration — Audit Listing

Sometimes you don't want similarity, you want *all* SEV1 incidents for a compliance review. `list_memory_records` with metadata filters gives you a deterministic audit list with no KNN involved.


In [ ]:
audit = data_client.list_memory_records(
    memoryId=memory_id,
    namespace=namespace,
    metadataFilters=[
        {"left": {"metadataKey": "incident_severity"}, "operator": "LESS_THAN_OR_EQUALS",
         "right": {"metadataValue": {"numberValue": 2}}}
    ],
)
print(f"All SEV1-SEV2 incidents ({len(audit.get('memoryRecordSummaries', []))}):")
for r in audit.get("memoryRecordSummaries", []):
    print(f"  - {r.get('content', {}).get('text', '')[:160]}")


## Step 11: Retrieval Tool for the Live Agent

Wrap the filtered retrieval in a Strands tool so the on-call agent can pull prior-incident context during a live page.


In [ ]:
@tool
def retrieve_prior_incidents(
    query: str,
    service: Optional[str] = None,
    max_severity: Optional[int] = None,
    mitigation: Optional[str] = None,
) -> str:
    """Retrieve prior incident episodes filtered by service, severity band, and mitigation type.

    Args:
        query: What are you looking for (free text, used for semantic relevance within the filtered set).
        service: Optional service name, e.g. 'auth-service'. Filters by services_touched CONTAINS <service>.
        max_severity: Optional maximum severity (1=worst, 5=trivial). Filters by incident_severity <= max_severity.
        mitigation: Optional mitigation type: revert, hotfix, rollback, config_change, scale_up, no_action.
    """
    filters = []
    if service:
        filters.append({"left": {"metadataKey": "services_touched"}, "operator": "CONTAINS",
                        "right": {"metadataValue": {"stringValue": service}}})
    if max_severity is not None:
        filters.append({"left": {"metadataKey": "incident_severity"}, "operator": "LESS_THAN_OR_EQUALS",
                        "right": {"metadataValue": {"numberValue": float(max_severity)}}})
    if mitigation:
        filters.append({"left": {"metadataKey": "mitigation_applied"}, "operator": "EQUALS_TO",
                        "right": {"metadataValue": {"stringValue": mitigation}}})
    search = {"searchQuery": query, "topK": 5}
    if filters:
        search["metadataFilters"] = filters
    r = data_client.retrieve_memory_records(
        memoryId=memory_id, namespace=namespace, searchCriteria=search
    )
    recs = r.get("memoryRecordSummaries", [])
    if not recs:
        return "no prior incidents matched"
    return "\n\n".join(f"- [{x.get('score', 0):.2f}] {x.get('content', {}).get('text', '')[:300]}" for x in recs)

logger.info("Agent retrieval tool ready")


## Step 12: Live On-Call Turn

The on-call engineer is paged: auth-service is throwing 401s again. The agent uses `retrieve_prior_incidents` with service and severity filters to surface the rollback we did last time.


In [ ]:
oncall_agent = Agent(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[check_service_health, list_recent_deploys, page_oncall, retrieve_prior_incidents],
    system_prompt=(
        "You are an on-call SRE assistant. When handling a live incident, FIRST check service health, "
        "THEN query prior incidents using retrieve_prior_incidents with narrow filters (service name, "
        "severity band, mitigation type). Quote the prior incident's root cause and mitigation so the "
        "engineer can decide whether the current symptoms match. Be concise — this is a live page."
    ),
)

page = (
    "PAGE: auth-service is throwing 401s for ~20% of users. "
    "Probably SEV1 if this keeps going. What did we do last time?"
)
response = oncall_agent(page)
print("\n=== AGENT RESPONSE ===\n", response)


## Step 13: Cleanup (Optional)


In [ ]:
# control_client.delete_memory(memoryId=memory_id)
# print(f"Deleted {memory_id}")


## What you built

- An **episodic** memory strategy with a metadata schema — showing that metadata works across all strategy types, not just semantic.
- A schema where **every field is FM-inferred** from conversation content (no event-time metadata supplied). The extractor populated SEV1-5 severity, a list of services touched, root-cause and mitigation categories, and an MTTR estimate.
- A `numericValidation` constraint on `incident_severity` (min 1, max 5) and `stringValidation` allowed-values on the category fields — demonstrating both validation types.
- **Compound retrieval** with 4 filters stacked: NUMBER `<=` + STRINGLIST `CONTAINS` + two STRING `EQUALS_TO`, applied before the KNN search runs.
- A live SRE agent whose retrieval is **scoped by operational dimensions the engineer actually cares about** during a page.
- Non-semantic enumeration (`list_memory_records`) for audit-style lookup.
